# Малый вертел: сегментация области + классификатор ошибки позиции

Ноутбук для Kaggle/Colab. Две модели (Шаг 2 плана, docs/plan.md):

1. **Сегментационная модель** — находит область малого вертела (полигон
   из CVAT, растеризованный в маску) на новом снимке.
2. **Классификатор ошибки позиции** — бинарный (есть нарушение критерия
   "позиция/ротация" или нет). Обучается на кропе ~1/5-1/4 от исходного
   снимка вокруг предсказанной области, с подсвеченной поверх тепловой
   картой вероятности сегментации (не бинарной маской по порогу 0.5 —
   вероятность передаётся как есть, неопределённость видна модели, а не
   отбрасывается). Истинная метка — из `разметка.xlsx`
   (hip_right_pos/hip_left_pos), не из самой геометрии.

В конце — ячейка, прогоняющая **обе модели последовательно**: сегментация
находит область → по её вероятностной карте вырезается кроп с подсветкой →
кроп идёт в классификатор. Это и есть реальный сценарий инференса (на
новых данных модель 1 не может использовать разметку человека, только
свой собственный прогноз) — причём классификатор ТРЕНИРУЕТСЯ на этом же
реалистичном сценарии (через OOF-предсказания сегментации), а не только
тестируется на нём.

## Перед запуском

Тот же `training_package` (images/), плюс `hip_keypoints_manifest.csv`
(там же координаты полигона малого вертела и истинная метка
`position_error`).

## Воспроизводимость

Те же меры, что в предыдущих ноутбуках: seed, детерминизм cuDNN,
group k-fold по study_uid (колонка `fold`), честный отложенный TEST_FOLD.


In [ ]:
!pip install -q albumentations timm scikit-learn


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


## Воспроизводимость: seed + детерминизм

In [ ]:
SEED = 42


def set_full_determinism(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_full_determinism(SEED)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_generator(seed: int = SEED) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


## Конфигурация

In [ ]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
IMAGES_DIR = "/kaggle/input/datasets/cgvhbjmk/dxa-data/images"
KEYPOINTS_MANIFEST_PATH = "/kaggle/input/datasets/cgvhbjmk/dxa-data/hip_keypoints_manifest.csv"
OUTPUT_DIR = "./outputs_lesser_trochanter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224  # рабочий кадр для сегментации (LongestMaxSize+Pad)
CROP_FRACTION = 0.22  # доля от большей стороны ИСХОДНОГО снимка для кропа классификатора
CLF_INPUT_SIZE = 128  # кроп ресайзится до этого размера перед классификатором
CROP_JITTER_FRAC = 0.15  # случайное смещение центра кропа при обучении (доля от размера кропа) — имитирует неточность реальной сегментации
BATCH_SIZE = 8
NUM_EPOCHS_SEG = 60
NUM_EPOCHS_CLF = 60
EARLY_STOPPING_PATIENCE = 15  # val_dice шумный на маленьких val-фолдах, не ловим случайный пик
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
N_FOLDS = 5
TEST_FOLD = N_FOLDS - 1
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Данные

Фильтруем строки с валидным полигоном (для сегментации) и отдельно —
с валидной меткой `position_error` (для классификатора). Это разные,
частично пересекающиеся подмножества.


In [ ]:
manifest = pd.read_csv(KEYPOINTS_MANIFEST_PATH)

seg_df = manifest[manifest["polygon"].notna() & (manifest["polygon"] != "")].reset_index(drop=True)
print(f"Для сегментации (есть полигон): {len(seg_df)}")

clf_df = seg_df[seg_df["position_error"].notna()].reset_index(drop=True)
clf_df = clf_df[clf_df["position_error"] != ""].reset_index(drop=True)
clf_df["position_error"] = clf_df["position_error"].astype(int)
print(f"Для классификатора (есть полигон + истинная метка position_error): {len(clf_df)}")
print(clf_df["position_error"].value_counts())

seg_cv = seg_df[seg_df["fold"] != TEST_FOLD].reset_index(drop=True)
seg_test = seg_df[seg_df["fold"] == TEST_FOLD].reset_index(drop=True)

clf_cv = clf_df[clf_df["fold"] != TEST_FOLD].reset_index(drop=True)
clf_test = clf_df[clf_df["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"\nSeg CV/test: {len(seg_cv)}/{len(seg_test)}")
print(f"Clf CV/test: {len(clf_cv)}/{len(clf_test)}")


## Общие утилиты: препроцессинг, полигон -> координаты, растеризация маски

Аугментация полигона идёт через тот же трюк, что и для keypoints —
вершины полигона трансформируются как список точек через
`keypoint_params`, а маска рисуется уже ПОСЛЕ геометрического
преобразования (не до), иначе растеризованная маска исказится при
интерполяции resize/rotate.


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def clahe(image: np.ndarray) -> np.ndarray:
    return cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(image)


def parse_polygon(text: str) -> list[tuple[float, float]]:
    return [tuple(map(float, p.split(","))) for p in text.split(";")]


def polygon_to_mask(size: int, polygon_xy: list[tuple[float, float]]) -> np.ndarray:
    mask = np.zeros((size, size), dtype=np.uint8)
    pts = np.array(polygon_xy, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def build_seg_train_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
        A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.4),
        A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), rotate=(-15, 15), fit_output=False, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False), seed=SEED)


def build_eval_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))


def to_normalized_tensor(image_u8: np.ndarray) -> torch.Tensor:
    rgb = np.stack([image_u8] * 3, axis=-1).astype(np.float32) / 255.0
    rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(rgb.transpose(2, 0, 1)).float()


## Кроп классификатора: пересчёт координат исходное разрешение <-> 224-кадр

Классификатор ошибки позиции работает не на микро-патче вокруг точки, а на
кропе ~1/5-1/4 от исходного снимка (`CROP_FRACTION`) с подсвеченной поверх
тепловой картой вероятности сегментации — так неточность сегментации
(Dice~0.57) не выбивает область из кадра целиком, а сама карта передаёт
классификатору неопределённость вместо жёсткого да/нет по порогу 0.5.

Сегментация предсказывает вероятность в 224x224 рабочем кадре
(LongestMaxSize+PadIfNeeded), а кроп для классификатора вырезается из
ИСХОДНОГО снимка (чтобы не терять детализацию и получить кроп предсказуемого
физического размера независимо от разрешения скана) — поэтому нужен точный
пересчёт координат/карты между этими двумя системами. Проверено отдельно на
синтетических данных перед отправкой в это обучение.


In [ ]:
def resize_pad_params(orig_h: int, orig_w: int, size: int = IMAGE_SIZE) -> dict:
    """Параметры LongestMaxSize(size)+PadIfNeeded(size,size,center) —
    нужны, чтобы пересчитывать координаты между исходным разрешением
    снимка и рабочим 224x224 кадром сегментации в обе стороны."""
    scale = size / max(orig_h, orig_w)
    new_h, new_w = orig_h * scale, orig_w * scale
    pad_top = (size - new_h) / 2
    pad_left = (size - new_w) / 2
    return {"scale": scale, "new_h": new_h, "new_w": new_w, "pad_top": pad_top, "pad_left": pad_left}


def prob_map_to_original(prob_224: np.ndarray, orig_h: int, orig_w: int, params: dict) -> np.ndarray:
    """Обрезает паддинг у 224x224 карты вероятностей и растягивает
    обратно до исходного разрешения — карта совпадает пиксель-в-пиксель
    с исходным снимком."""
    top, left = int(round(params["pad_top"])), int(round(params["pad_left"]))
    h, w = int(round(params["new_h"])), int(round(params["new_w"]))
    sub = prob_224[top:top + h, left:left + w]
    return cv2.resize(sub, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)


def soft_centroid(prob_map: np.ndarray, eps: float = 1e-6) -> tuple[float, float]:
    """Взвешенный центроид по ВСЕЙ вероятностной карте, без порога —
    всегда определён, в отличие от порога 0.5 (тот может дать пустую
    маску, если уверенность нигде его не превышает)."""
    h, w = prob_map.shape
    ys, xs = np.mgrid[0:h, 0:w]
    total = prob_map.sum() + eps
    cx = float((xs * prob_map).sum() / total)
    cy = float((ys * prob_map).sum() / total)
    return cx, cy


@torch.no_grad()
def predict_seg_prob_original(image_u8: np.ndarray, models: list) -> np.ndarray:
    """image_u8 — уже CLAHE'нутый снимок в исходном разрешении.
    Прогоняет через (ансамбль) сегментационных моделей, возвращает карту
    вероятностей в исходном разрешении."""
    orig_h, orig_w = image_u8.shape
    params = resize_pad_params(orig_h, orig_w)

    image_224 = build_eval_transform()(image=image_u8, keypoints=[])["image"]
    tensor = to_normalized_tensor(image_224).unsqueeze(0).to(DEVICE)

    probs = torch.stack([torch.sigmoid(m(tensor)) for m in models], dim=0).mean(dim=0)
    prob_224 = probs[0, 0].cpu().numpy()
    return prob_map_to_original(prob_224, orig_h, orig_w, params)


def crop_around_point(image_u8: np.ndarray, prob_map: np.ndarray, cx: float, cy: float,
                       fraction: float, out_size: int, jitter_frac: float = 0.0):
    """Вырезает квадрат вокруг (cx, cy) размером `fraction` от большей
    стороны снимка — и саму картинку, и выровненную с ней карту
    вероятностей — и ресайзит обе до out_size. jitter_frac>0 (только
    train) случайно смещает центр — имитирует неточность реальной
    сегментации на инференсе, иначе классификатор учится только на
    идеально отцентрованных вырезках."""
    h, w = image_u8.shape
    crop_size = int(round(fraction * max(h, w)))

    if jitter_frac > 0:
        cx = cx + random.uniform(-jitter_frac, jitter_frac) * crop_size
        cy = cy + random.uniform(-jitter_frac, jitter_frac) * crop_size

    half = crop_size // 2
    x0, y0 = int(round(cx - half)), int(round(cy - half))

    img_pad = cv2.copyMakeBorder(image_u8, half, half, half, half, cv2.BORDER_CONSTANT, value=0)
    prob_pad = cv2.copyMakeBorder(prob_map, half, half, half, half, cv2.BORDER_CONSTANT, value=0)
    x0p, y0p = x0 + half, y0 + half

    img_crop = img_pad[y0p:y0p + crop_size, x0p:x0p + crop_size]
    prob_crop = prob_pad[y0p:y0p + crop_size, x0p:x0p + crop_size]

    img_crop = cv2.resize(img_crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    prob_crop = cv2.resize(prob_crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return img_crop, prob_crop


def heatmap_overlay(gray_u8: np.ndarray, heat01: np.ndarray, alpha: float = 0.6) -> np.ndarray:
    """Красит область высокой вероятности тёплым оттенком поверх ч/б
    снимка. R/G/B всё ещё коррелируют с реальной яркостью пикселей (в
    отличие от отдельного «сырого» 4-го канала) — предобученные веса
    conv1 resnet18 (они заморожены) продолжают видеть похожее на фото
    изображение, а не произвольный новый канал."""
    gray_f = gray_u8.astype(np.float32)
    heat = np.clip(heat01, 0.0, 1.0)
    r = np.clip(gray_f + heat * 255.0 * alpha, 0, 255)
    g = gray_f
    b = np.clip(gray_f - heat * 255.0 * alpha * 0.5, 0, 255)
    return np.stack([r, g, b], axis=-1).astype(np.uint8)


def to_normalized_tensor_rgb(rgb_u8: np.ndarray) -> torch.Tensor:
    rgb = rgb_u8.astype(np.float32) / 255.0
    rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(rgb.transpose(2, 0, 1)).float()


## Модель 1 — сегментация (encoder-decoder, 1 канал, sigmoid)

In [ ]:
class UpBlock(nn.Module):
    """Upsample x2 -> concat со skip-connection encoder'а -> conv+BN+ReLU x2.

    Без skip-connections декодер восстанавливает маску только из
    боттлнека 7x7 — для такой маленькой структуры, как малый вертел,
    этого недостаточно (проверено: Dice ~0.15, сегментация часто вообще
    ничего не находит). Skip-connections возвращают декодеру
    пространственную детализацию с ранних слоёв энкодера — стандартное
    решение именно этой проблемы (U-Net).
    """

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch + skip_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, skip):
        x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


class SegmentationModel(nn.Module):
    # Каналы/разрешения encoder-выходов resnet18 при входе 224x224
    # (проверено эмпирически): stage0=64@112, stage1=64@56, stage2=128@28,
    # stage3=256@14, stage4=512@7 (боттлнек).
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            "resnet18", pretrained=True, features_only=True, out_indices=(0, 1, 2, 3, 4)
        )

        for name, param in self.encoder.named_parameters():
            # для сегментации замораживаем больше слоёв, чем в классификаторе —
            # датасет тут заметно меньше (~90-100 img/fold), обучаемый только
            # layer4 снижает риск переобучения (train/val Dice расходились)
            if name.startswith(("conv1", "bn1", "layer1", "layer2", "layer3")):
                param.requires_grad = False

        self.up1 = UpBlock(in_ch=512, skip_ch=256, out_ch=256)  # 7 -> 14
        self.up2 = UpBlock(in_ch=256, skip_ch=128, out_ch=128)  # 14 -> 28
        self.up3 = UpBlock(in_ch=128, skip_ch=64, out_ch=64)    # 28 -> 56
        self.up4 = UpBlock(in_ch=64, skip_ch=64, out_ch=32)     # 56 -> 112

        # финальный подъём до 224x224 — skip-а на этом разрешении у
        # resnet18 нет (stem сразу даёт 112), просто апсемпл + conv
        self.up5 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        feat0, feat1, feat2, feat3, feat4 = self.encoder(x)
        d = self.up1(feat4, feat3)
        d = self.up2(d, feat2)
        d = self.up3(d, feat1)
        d = self.up4(d, feat0)
        d = self.up5(d)
        return self.head(d)  # логиты, sigmoid снаружи


def dice_loss(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


class SegDataset(Dataset):
    def __init__(self, df: pd.DataFrame, images_dir: str, train: bool):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, row["image_filename"])
        image_u8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        image_u8 = clahe(image_u8)
        polygon = parse_polygon(row["polygon"])

        transform = build_seg_train_transform() if self.train else build_eval_transform()
        transformed = transform(image=image_u8, keypoints=polygon)
        image_t = transformed["image"]
        polygon_t = transformed["keypoints"]

        mask = polygon_to_mask(IMAGE_SIZE, polygon_t)

        image_tensor = to_normalized_tensor(image_t)
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(0)
        return image_tensor, mask_tensor


## Обучение сегментации (CV по фолдам, кроме TEST_FOLD)

In [ ]:
def build_optimizer(model: nn.Module):
    decoder_modules = [model.up1, model.up2, model.up3, model.up4, model.up5, model.head]
    head_params = [p for m in decoder_modules for p in m.parameters()]
    backbone_params = [p for p in model.encoder.parameters() if p.requires_grad]
    # weight_decay повышен относительно дефолта (0.01) — маленький датасет,
    # сильнее давим на переобучение регуляризацией, раз train/val Dice расходились
    return torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=0.05)


def run_seg_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_dice, n = 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss = dice_loss(logits, masks) + nn.functional.binary_cross_entropy_with_logits(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            with torch.no_grad():
                pred = (torch.sigmoid(logits) > 0.5).float()
                inter = (pred * masks).sum(dim=(1, 2, 3))
                union = pred.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
                dice = ((2 * inter + 1e-6) / (union + 1e-6)).mean().item()

            bs = images.size(0)
            total_loss += loss.item() * bs
            total_dice += dice * bs
            n += bs

    return total_loss / n, total_dice / n


def train_seg_one_fold(fold: int, seg_cv: pd.DataFrame):
    set_full_determinism(int(SEED + fold))
    train_df = seg_cv[seg_cv["fold"] != fold]
    val_df = seg_cv[seg_cv["fold"] == fold]

    train_ds = SegDataset(train_df, IMAGES_DIR, train=True)
    val_ds = SegDataset(val_df, IMAGES_DIR, train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                               drop_last=True, worker_init_fn=worker_init_fn, generator=make_generator(int(SEED + fold)))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = SegmentationModel().to(DEVICE)
    optimizer = build_optimizer(model)

    best_dice, best_state, patience = -1.0, None, 0
    for epoch in range(NUM_EPOCHS_SEG):
        train_loss, train_dice = run_seg_epoch(model, train_loader, optimizer)
        val_loss, val_dice = run_seg_epoch(model, val_loader)
        print(f"[seg] fold {fold} | epoch {epoch+1}/{NUM_EPOCHS_SEG} | "
              f"train_dice={train_dice:.3f} | val_dice={val_dice:.3f}")

        if val_dice > best_dice:
            best_dice = val_dice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print(f"[seg] fold {fold}: early stopping на эпохе {epoch+1}")
                break

    torch.save({"model_state_dict": best_state, "image_size": IMAGE_SIZE, "fold": fold, "best_val_dice": best_dice},
               os.path.join(OUTPUT_DIR, f"seg_fold{fold}.pt"))
    return best_dice


seg_fold_scores = []
for fold in sorted(int(f) for f in seg_cv["fold"].unique()):
    score = train_seg_one_fold(fold, seg_cv)
    seg_fold_scores.append(score)
    print(f"=== seg fold {fold}: best val Dice = {score:.3f} ===\n")

print("Dice по фолдам:", [round(s, 3) for s in seg_fold_scores])
print("Среднее:", round(np.mean(seg_fold_scores), 3), "± std:", round(np.std(seg_fold_scores), 3))


## Модели сегментации по фолдам — нужны для честной OOF-разметки классификатора

Для строки из фолда `k` при обучении классификатора карту вероятности
должна строить модель, которая **фолд k не видела на своём обучении**
(`seg_fold{k}.pt`, где k — её собственный val-фолд) — иначе классификатор
учился бы на нереалистично чистых картах и не умел бы работать с шумом,
какой будет на реальном инференсе.


In [ ]:
seg_models_by_fold: dict[int, nn.Module] = {}
for fold in sorted(int(f) for f in seg_cv["fold"].unique()):
    ckpt = torch.load(os.path.join(OUTPUT_DIR, f"seg_fold{fold}.pt"), map_location=DEVICE, weights_only=False)
    m = SegmentationModel().to(DEVICE)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    seg_models_by_fold[fold] = m

print(f"Загружено моделей сегментации: {len(seg_models_by_fold)}")


## Модель 2 — классификатор ошибки позиции (на кропе с тепловой картой)

Кроп и карта вероятности строятся через OOF-предсказание своей же
сегментационной модели (без утечки — см. markdown выше) уже и при
обучении, не только на инференсе. Это и есть честная тренировка на
реалистично несовершённых данных, а не на идеальной разметке.


In [ ]:
class LesserTrochanterClfDataset(Dataset):
    """Классификатор смотрит не на изолированный микро-патч, а на кроп
    ~1/5-1/4 от исходного снимка с подсвеченной поверх тепловой картой
    вероятности малого вертела (см. markdown-ячейку выше про
    CROP_FRACTION/heatmap_overlay).

    При обучении карта вероятности берётся из OOF-предсказания
    сегментации (seg_models_by_fold[row.fold] — модель, НЕ видевшая этот
    снимок на своём обучении), а не из разметки: иначе классификатор
    видел бы только идеально чистые карты и не умел бы работать с
    реалистичным шумом инференса."""

    def __init__(self, df: pd.DataFrame, images_dir: str, seg_models_by_fold: dict, train: bool):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.seg_models_by_fold = seg_models_by_fold
        self.train = train
        self.aug = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
            A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        ]) if train else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, row["image_filename"])
        image_u8 = clahe(cv2.imread(path, cv2.IMREAD_GRAYSCALE))

        model = self.seg_models_by_fold[int(row["fold"])]
        prob_map = predict_seg_prob_original(image_u8, [model])
        cx, cy = soft_centroid(prob_map)

        jitter = CROP_JITTER_FRAC if self.train else 0.0
        img_crop, prob_crop = crop_around_point(image_u8, prob_map, cx, cy, CROP_FRACTION, CLF_INPUT_SIZE, jitter)

        if self.aug is not None:
            img_crop = self.aug(image=img_crop)["image"]

        rgb = heatmap_overlay(img_crop, prob_crop)
        tensor = to_normalized_tensor_rgb(rgb)
        label = int(row["position_error"])
        return tensor, label


class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("resnet18", pretrained=True, num_classes=2)
        for name, param in self.backbone.named_parameters():
            if name.startswith(("conv1", "bn1", "layer1")):
                param.requires_grad = False

    def forward(self, x):
        return self.backbone(x)


## Обучение классификатора (с учётом дисбаланса классов)

In [ ]:
def run_clf_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader.dataset), f1, all_preds, all_labels


def train_clf_one_fold(fold: int, clf_cv: pd.DataFrame):
    set_full_determinism(int(SEED + fold))
    train_df = clf_cv[clf_cv["fold"] != fold]
    val_df = clf_cv[clf_cv["fold"] == fold]

    if val_df["position_error"].nunique() < 2:
        print(f"[clf] fold {fold}: в валидации только один класс, пропуск фолда (слишком мало данных)")
        return None

    # class weights — датасет маленький и несбалансированный (мало violation-примеров)
    counts = train_df["position_error"].value_counts()
    weights = torch.tensor([1.0 / counts.get(0, 1), 1.0 / counts.get(1, 1)], dtype=torch.float32)
    weights = weights / weights.sum() * 2

    train_ds = LesserTrochanterClfDataset(train_df, IMAGES_DIR, seg_models_by_fold, train=True)
    val_ds = LesserTrochanterClfDataset(val_df, IMAGES_DIR, seg_models_by_fold, train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                               drop_last=True, worker_init_fn=worker_init_fn, generator=make_generator(int(SEED + fold)))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = SmallClassifier().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR)
    criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))

    best_f1, best_state, patience = -1.0, None, 0
    for epoch in range(NUM_EPOCHS_CLF):
        train_loss, train_f1, _, _ = run_clf_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, val_preds, val_labels = run_clf_epoch(model, val_loader, criterion)
        print(f"[clf] fold {fold} | epoch {epoch+1}/{NUM_EPOCHS_CLF} | "
              f"train_f1={train_f1:.3f} | val_f1={val_f1:.3f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print(f"[clf] fold {fold}: early stopping на эпохе {epoch+1}")
                break

    torch.save({"model_state_dict": best_state, "fold": fold, "best_val_f1": best_f1},
               os.path.join(OUTPUT_DIR, f"clf_fold{fold}.pt"))
    return best_f1


clf_fold_scores = []
for fold in sorted(int(f) for f in clf_cv["fold"].unique()):
    score = train_clf_one_fold(fold, clf_cv)
    if score is not None:
        clf_fold_scores.append(score)
        print(f"=== clf fold {fold}: best val macro-F1 = {score:.3f} ===\n")

print("Macro-F1 по фолдам:", [round(s, 3) for s in clf_fold_scores])
if clf_fold_scores:
    print("Среднее:", round(np.mean(clf_fold_scores), 3), "± std:", round(np.std(clf_fold_scores), 3))
print("\nЧЕСТНАЯ ОГОВОРКА: датасет для этого классификатора очень маленький "
      f"({len(clf_cv)} изображений на CV, дисбаланс классов) — метрики "
      "нестабильны между фолдами, ожидаемо на таком объёме.")


## Финальная проверка: обе модели вместе, на отложенном тесте

Ансамбль сегментации → мягкий центроид по всей карте вероятности (без
порога 0.5, см. выше) → кроп ~1/5-1/4 снимка с тепловой картой поверх →
ансамбль классификатора. Это честная проверка всей цепочки целиком, как
она будет работать на реальных новых данных.


In [ ]:
seg_models = list(seg_models_by_fold.values())

clf_models = []
for fold in sorted(int(f) for f in clf_cv["fold"].unique()):
    ckpt_path = os.path.join(OUTPUT_DIR, f"clf_fold{fold}.pt")
    if not os.path.exists(ckpt_path):
        continue
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m = SmallClassifier().to(DEVICE)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    clf_models.append(m)

print(f"Моделей сегментации в ансамбле: {len(seg_models)}, классификации: {len(clf_models)}")


@torch.no_grad()
def predict_pipeline(image_u8: np.ndarray) -> dict:
    """image_u8 — уже CLAHE'нутый снимок в исходном разрешении.
    Честный полный сценарий инференса: ансамбль сегментации -> мягкий
    центроид по всей вероятностной карте (без порога 0.5 — иначе на
    неуверенных снимках маска может оказаться пустой и мы теряем
    предсказание вообще) -> кроп ~1/5-1/4 снимка вокруг центроида с
    подсвеченной поверх тепловой картой -> ансамбль классификатора."""
    prob_map = predict_seg_prob_original(image_u8, seg_models)
    cx, cy = soft_centroid(prob_map)
    img_crop, prob_crop = crop_around_point(image_u8, prob_map, cx, cy, CROP_FRACTION, CLF_INPUT_SIZE, jitter_frac=0.0)
    rgb = heatmap_overlay(img_crop, prob_crop)
    tensor = to_normalized_tensor_rgb(rgb).unsqueeze(0).to(DEVICE)

    clf_probs = torch.stack([torch.softmax(m(tensor), dim=1) for m in clf_models], dim=0).mean(dim=0)
    pred_label = int(clf_probs.argmax(dim=1).item())
    confidence = float(clf_probs[0, pred_label].item())

    return {"position_error": pred_label, "confidence": confidence,
            "seg_confidence": float(prob_map.max())}


# Прогон на отложенном тесте (clf_test — там, где есть и полигон, и истинная метка).
# Порога больше нет — предсказание есть всегда, но отдельно отмечаем снимки
# с низкой пиковой уверенностью сегментации (не как причину пропуска, а как
# диагностику: центроид на таких снимках менее надёжен).
test_preds, test_labels, low_conf = [], [], []
for _, row in clf_test.iterrows():
    path = os.path.join(IMAGES_DIR, row["image_filename"])
    image_u8 = clahe(cv2.imread(path, cv2.IMREAD_GRAYSCALE))
    result = predict_pipeline(image_u8)
    if result["seg_confidence"] < 0.3:
        low_conf.append(row["image_filename"])
    test_preds.append(result["position_error"])
    test_labels.append(int(row["position_error"]))

print(f"\nОтложенный тест целиком через пайплайн: {len(test_preds)} изображений")
if low_conf:
    print(f"Снимков с низкой пиковой уверенностью сегментации (<0.3): {len(low_conf)} — {low_conf}")

if len(set(test_labels)) > 1:
    print(classification_report(test_labels, test_preds, target_names=["норма", "ошибка_позиции"], digits=3))
    cm = confusion_matrix(test_labels, test_preds)
    print("Confusion matrix:")
    print(pd.DataFrame(cm, index=["норма", "ошибка_позиции"], columns=["норма", "ошибка_позиции"]))
else:
    print("В отложенном тесте только один класс — честную полноценную метрику посчитать нельзя, "
          "слишком мало данных с нарушением в этом фолде.")
    print("accuracy:", accuracy_score(test_labels, test_preds))


## Что дальше

- `./outputs_lesser_trochanter/seg_fold{N}.pt` — веса сегментации
- `./outputs_lesser_trochanter/clf_fold{N}.pt` — веса классификатора
- Честная оговорка: классификатор ошибки позиции обучен на очень
  маленьком и несбалансированном наборе (десятки изображений с
  нарушением на весь датасет) — метрики нестабильны, воспринимать как
  первый прототип, не финальное решение. Больше данных даст либо
  доразметка, либо доступ к дополнительным DXA-датасетам (обсуждали
  раньше — UK Biobank, VerteNet и т.д.)
- Если и с более крупным кропом + тепловой картой классификатор всё ещё
  не ловит редкий класс на честном тесте — это, скорее всего, уже предел
  именно ОБЪЁМА данных (единицы-десятки положительных примеров), а не
  архитектуры или разметки признака; тут поможет только больше размеченных
  случаев с нарушением позиции.
